# GR00T N1.6 — Step 3: Evaluation

This notebook evaluates the fine-tuned GR00T model locally using open-loop evaluation.

**Prerequisites:**
- Run `01_data_preparation.ipynb` (local dataset at `./datasets/bridge_lerobot`)
- Run `02_training_job.ipynb` and download the model artifacts
- GPU instance with 48GB+ VRAM (g6e.48xlarge or p4d.24xlarge)

**What this does:**
1. Sets up Isaac-GR00T locally for evaluation
2. Evaluates the fine-tuned checkpoint on training data
3. Evaluates the base model for comparison
4. Creates a held-out test set and evaluates generalization
5. Compares results

## 1. Environment Setup

Clone Isaac-GR00T and install dependencies. Only needed once per instance.

In [1]:
%%bash
set -e

# === System deps ===
sudo apt-get update -qq && sudo apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 git-lfs ffmpeg 2>/dev/null || true

# === Pin torch+torchvision (MUST use PyTorch index to avoid getting torch 2.11+) ===
pip install -q torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu124
pip install -q optree==0.13.1 --force-reinstall --no-deps

# === Install other deps (do NOT reinstall torch here) ===
pip install -q \
    'transformers>=4.40.0,<4.52' \
    'huggingface-hub>=0.30.0,<1.0' \
    'pyarrow>=14.0' \
    'av' \
    'opencv-python-headless' \
    'datasets' \
    'accelerate' \
    'tyro' \
    'albumentations==1.4.18'

# === Install flash-attn (build from source against local torch) ===
pip install flash-attn==2.7.4.post1 --no-cache-dir --no-build-isolation 2>/dev/null || echo '[WARN] flash-attn build failed'

# === Clone Isaac-GR00T (n1.6-release for GR00T-N1.6-3B) ===
if [ -d "Isaac-GR00T" ]; then
    cd Isaac-GR00T
    CURRENT_TAG=$(git describe --tags --exact-match 2>/dev/null || echo 'none')
    if [ "$CURRENT_TAG" != "n1.6-release" ]; then
        echo '[INFO] Switching to n1.6-release...'
        git fetch --tags
        git checkout n1.6-release
        git submodule update --init --recursive
    fi
else
    git clone --recurse-submodules --branch n1.6-release https://github.com/NVIDIA/Isaac-GR00T.git
    cd Isaac-GR00T
fi
git submodule update --init --recursive

# === Patch and install GR00T ===
PYPROJECT="pyproject.toml"
cp "$PYPROJECT" "${PYPROJECT}.bak" 2>/dev/null || true
sed -i '/tensorrt/d' "$PYPROJECT"
sed -i '/onnx/d' "$PYPROJECT"
sed -i 's/requires-python.*==3\.10.*/requires-python = ">=3.10"/' "$PYPROJECT"
pip install -e . --no-build-isolation 2>&1 | tail -3
mv "${PYPROJECT}.bak" "$PYPROJECT" 2>/dev/null || true

# === Force huggingface-hub<1.0 AFTER GR00T install (GR00T pulls in >=1.0) ===
pip install -q 'huggingface-hub>=0.30.0,<1.0' --force-reinstall --no-deps

# === Verify ===
python3 -c "import gr00t; print('gr00t OK')"
python3 -c "import flash_attn; print(f'flash_attn {flash_attn.__version__} OK')"
python3 -c "import torch; print(f'torch {torch.__version__} OK')"
python3 -c "import torchvision; print(f'torchvision {torchvision.__version__} OK')"
echo '=== Setup complete ==='

      Successfully uninstalled gr00t-0.1.0


gr00t OK
flash_attn 2.7.4.post1 OK
torch 2.7.1+cu126 OK
torchvision 0.22.1+cu126 OK
=== Setup complete ===


In [2]:
from getpass import getpass
from huggingface_hub import login

# Prompt for Hugging Face token (input is hidden)
hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

# Download base model weights
from huggingface_hub import snapshot_download
path = snapshot_download("nvidia/GR00T-N1.6-3B")
print(f"Base model at: {path}")

Enter your Hugging Face token:  ········


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Base model at: /home/sagemaker-user/.cache/huggingface/hub/models--nvidia--GR00T-N1.6-3B/snapshots/d0814e7ecb19202e7c8468b46098b0b7ef3a6d61


## 2. Configuration

In [3]:
import os, glob, boto3

# Auto-detect the latest completed GR00T training job
sm_client = boto3.client('sagemaker')
job_resp = sm_client.list_training_jobs(
    NameContains='groot-n16-finetune-bridge',
    SortBy='CreationTime', SortOrder='Descending', MaxResults=10)
completed_jobs = [j for j in job_resp['TrainingJobSummaries'] if j['TrainingJobStatus'] == 'Completed']
if completed_jobs:
    TRAINING_JOB_NAME = completed_jobs[0]['TrainingJobName']
    print(f'Auto-detected training job: {TRAINING_JOB_NAME}')
else:
    TRAINING_JOB_NAME = 'UNKNOWN'
    print('WARNING: No completed training jobs found. Set TRAINING_JOB_NAME manually.')

# Path to the fine-tuned checkpoint
CHECKPOINT_PATH = str(os.path.abspath(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/bridge_finetune/checkpoint-2000'))

# Fallback: search for any checkpoint dir if exact path doesn't exist
if not os.path.isdir(CHECKPOINT_PATH):
    candidates = glob.glob(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/**/checkpoint-*', recursive=True)
    if candidates:
        CHECKPOINT_PATH = str(os.path.abspath(candidates[0]))
        print(f'Using checkpoint: {CHECKPOINT_PATH}')
    else:
        print(f'WARNING: No checkpoint found. Contents:')
        extracted = f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/'
        if os.path.isdir(extracted):
            for root, dirs, files in os.walk(extracted):
                for d in dirs[:10]:
                    print(f'  {os.path.join(root, d)}')

# Dataset paths
TRAIN_DATASET = "./datasets/bridge_lerobot"
TEST_DATASET = "./datasets/bridge_lerobot_test"

# Evaluation params
TRAJ_IDS = "0 1 2 3 4"
ACTION_HORIZON = 16

# Modality config
MODALITY_CONFIG = "./scripts/utils/bridge_modality_config.py"

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Train dataset: {TRAIN_DATASET}")
print(f"Test dataset: {TEST_DATASET}")

Auto-detected training job: groot-n16-finetune-bridge-20260417193941
Checkpoint: /home/sagemaker-user/GR00T_SMTJ/model_artifacts/groot-n16-finetune-bridge-20260417193941/extracted/bridge_finetune/checkpoint-2000
Train dataset: ./datasets/bridge_lerobot
Test dataset: ./datasets/bridge_lerobot_test


## 3. Evaluate Fine-Tuned Model (Training Data)

In [4]:
import os
TRAIN_DATASET = os.path.abspath("./datasets/bridge_lerobot")
TEST_DATASET = os.path.abspath("./datasets/bridge_lerobot_test")
os.environ["TRAIN_DATASET"] = TRAIN_DATASET
os.environ["TEST_DATASET"] = TEST_DATASET
print(f"TRAIN_DATASET = '{TRAIN_DATASET}'")


TRAIN_DATASET = '/home/sagemaker-user/GR00T_SMTJ/datasets/bridge_lerobot'


In [5]:
import os
os.environ["CHECKPOINT_PATH"] = CHECKPOINT_PATH
os.environ["TRAIN_DATASET"] = TRAIN_DATASET
os.environ["TEST_DATASET"] = TEST_DATASET
os.environ["TRAJ_IDS"] = TRAJ_IDS
os.environ["ACTION_HORIZON"] = str(ACTION_HORIZON)


In [6]:
%%bash
cd Isaac-GR00T
echo "=== Evaluating fine-tuned model on training data ==="
python gr00t/eval/open_loop_eval.py \
    --dataset-path "$TRAIN_DATASET" \
    --embodiment-tag NEW_EMBODIMENT \
    --model-path "$CHECKPOINT_PATH" \
    --traj-ids $TRAJ_IDS \
    --action-horizon $ACTION_HORIZON
echo "=== Done ==="

=== Evaluating fine-tuned model on training data ===


2026-04-20 19:01:04.441430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776711664.454188   25071 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776711664.458166   25071 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-20 19:01:04.470920: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:root:Extracted global_step 2000 from checkpoint path
/opt/conda/lib/python3.12/site-packages/albumenta

Tune backbone llm: False
Tune backbone visual: True
Backbone trainable parameter: model.vision_model.vision_model.embeddings.patch_embedding.weight
Backbone trainable parameter: model.vision_model.vision_model.embeddings.patch_embedding.bias
Backbone trainable parameter: model.vision_model.vision_model.embeddings.position_embedding.weight
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.layer_norm1.weight
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.layer_norm1.bias
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.self_attn.k_proj.weight
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.self_attn.k_proj.bias
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.self_attn.v_proj.weight
Backbone trainable parameter: model.vision_model.vision_model.encoder.layers.0.self_attn.v_proj.bias
Backbone trainable parameter: model.vision_model.vision_m

INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:set VIDEO_TOTAL_PIXELS: 90316800
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.
INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:Processor Eagle3_VLProcessor:
- image_processor: Eagle3_VLImageProcessorFast {
  "auto_map": {
    "AutoImageProcessor": "image_processing_eagle3_vl_fast.Eagle3_VLImageProcessorFast",
    "AutoProcessor": "processing_eagle3_vl.Eagle3_VLProcessor"
  },
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": false,
  "device": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": false,
  "do_rescale": true,
  "do_resize": false,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Eagle3_VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "input_data_format": null,
  "processor_class": "Eagle3_VLP


Casting trainable parameter model.language_model.model.layers.15.self_attn.o_proj.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.self_attn.q_norm.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.self_attn.k_norm.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.mlp.gate_proj.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.mlp.up_proj.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.mlp.down_proj.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.input_layernorm.weight to fp32
Casting trainable parameter model.language_model.model.layers.15.post_attention_layernorm.weight to fp32
Casting trainable parameter model.mlp1.0.weight to fp32
Casting trainable parameter model.mlp1.0.bias to fp32
Casting trainable parameter model.mlp1.1.weight to fp32
Casting trainable parameter model.mlp1.1.bias to fp32
Casting

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.54it/s]
INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:Processor Eagle3_VLProcessor:
- image_processor: Eagle3_VLImageProcessorFast {
  "auto_map": {
    "AutoImageProcessor": "image_processing_eagle3_vl_fast.Eagle3_VLImageProcessorFast",
    "AutoProcessor": "processing_eagle3_vl.Eagle3_VLProcessor"
  },
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": false,
  "device": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": false,
  "do_rescale": true,
  "do_resize": false,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Eagle3_VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "input_data_format": null,
  "processor_class": "Eagle3_VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "height": 448,
    "width": 448
  }
}



=== Done ===


## 4. Evaluate Base Model (for comparison)

The base model doesn't know about our custom embodiment, so we monkey-patch the processor.

In [7]:
import sys
sys.path.insert(0, "Isaac-GR00T")

# Build bridge modality config
from gr00t.data.types import ModalityConfig, ActionConfig, ActionRepresentation, ActionType, ActionFormat

bridge_modality = {
    "video": ModalityConfig(delta_indices=[0], modality_keys=["front"]),
    "state": ModalityConfig(delta_indices=[0], modality_keys=["arm"]),
    "action": ModalityConfig(
        delta_indices=list(range(0, 16)),
        modality_keys=["arm"],
        action_configs=[ActionConfig(
            rep=ActionRepresentation.ABSOLUTE,
            type=ActionType.NON_EEF,
            format=ActionFormat.DEFAULT,
        )],
    ),
    "language": ModalityConfig(delta_indices=[0], modality_keys=["annotation.human.task_description"]),
}

# Monkey-patch Gr00tPolicy
from gr00t.policy.gr00t_policy import Gr00tPolicy
import torch
import numpy as np
from pathlib import Path
from transformers import AutoModel, AutoProcessor

def _patched_init(self, embodiment_tag, model_path, *, device, strict=True):
    super(Gr00tPolicy, self).__init__(strict=strict)
    model_dir = Path(model_path)
    model = AutoModel.from_pretrained(model_dir, trust_remote_code=True)
    model.eval()
    model.to(device=device, dtype=torch.bfloat16)
    self.model = model
    self.processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    self.processor.eval()

    tag = embodiment_tag.value
    configs = self.processor.get_modality_configs()
    if tag not in configs:
        configs[tag] = bridge_modality

    sap = self.processor.state_action_processor
    if hasattr(sap, 'modality_configs') and tag not in sap.modality_configs:
        sap.modality_configs[tag] = bridge_modality

    if hasattr(sap, 'norm_params') and tag not in sap.norm_params:
        arm_stats = {
            'min': np.array([-0.0002622, -0.00387925, 0., 0., 0., 0., 0.]),
            'max': np.array([0.0006804, 0.00021, 0., 0., 0., 0., 0.]),
            'dim': np.array(7),
            'mean': np.array([-6.32212107e-07, 2.15603291e-06, 0., 0., 0., 0., 0.]),
            'std': np.array([2.25099247e-05, 3.73347902e-05, 0., 0., 0., 0., 0.]),
        }
        sap.norm_params[tag] = {
            'state': {'arm': dict(arm_stats)},
            'action': {'arm': dict(arm_stats)},
        }

    self.embodiment_tag = embodiment_tag
    self.modality_configs = configs[embodiment_tag.value]
    self.collate_fn = self.processor.collator
    language_keys = self.modality_configs["language"].modality_keys
    language_delta_indices = self.modality_configs["language"].delta_indices
    assert len(language_keys) == 1
    assert len(language_delta_indices) == 1
    self.language_key = language_keys[0]

Gr00tPolicy.__init__ = _patched_init
print("Gr00tPolicy patched for base model evaluation.")

2026-04-20 19:01:32.587131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776711692.599420   24613 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776711692.603294   24613 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-20 19:01:32.616633: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Gr00tPolicy patched for base model evaluation.


In [8]:
# Force GR00T model registration in the current process
import gr00t.model  # This registers Gr00tN1d6 with transformers AutoModel

import runpy, sys
import gr00t.policy.gr00t_policy as gp
gp.Gr00tPolicy.__init__ = _patched_init

traj_ids = TRAJ_IDS.split()
sys.argv = [
    "open_loop_eval.py",
    "--dataset-path", TRAIN_DATASET,
    "--embodiment-tag", "NEW_EMBODIMENT",
    "--model-path", "nvidia/GR00T-N1.6-3B",
    "--traj-ids",
] + traj_ids + [
    "--action-horizon", str(ACTION_HORIZON),
]

print("Running base model evaluation...")
runpy.run_path("Isaac-GR00T/gr00t/eval/open_loop_eval.py", run_name="__main__")
print("Base model evaluation complete.")


/opt/conda/bin/../libexec/gcc/x86_64-conda-linux-gnu/13.4.0/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/conda/bin/../libexec/gcc/x86_64-conda-linux-gnu/13.4.0/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/conda/lib/python3.12/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Running base model evaluation...


INFO:transformers_modules.Eagle-Block2A-2B-v2.modeling_eagle3_vl:mlp_checkpoint: False


Tune backbone llm: False
Tune backbone visual: False
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.q_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.k_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.v_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.o_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.q_norm.weight
Backbone trainable parameter: model.language_model.model.layers.12.self_attn.k_norm.weight
Backbone trainable parameter: model.language_model.model.layers.12.mlp.gate_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.mlp.up_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.mlp.down_proj.weight
Backbone trainable parameter: model.language_model.model.layers.12.input_layernorm.weight
Backbone trainable parameter: model.language_mode

INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:set VIDEO_TOTAL_PIXELS: 90316800
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


Tune action head projector: True
Tune action head diffusion model: True
Tune action head vlln: True


INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:Processor Eagle3_VLProcessor:
- image_processor: Eagle3_VLImageProcessorFast {
  "auto_map": {
    "AutoImageProcessor": "image_processing_eagle3_vl_fast.Eagle3_VLImageProcessorFast",
    "AutoProcessor": "processing_eagle3_vl.Eagle3_VLProcessor"
  },
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": false,
  "device": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": false,
  "do_rescale": true,
  "do_resize": false,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Eagle3_VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "input_data_format": null,
  "processor_class": "Eagle3_VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "height": 448,
    "width": 448
  }
}

- tokenizer: Qwen2TokenizerFast(name_or_path='/home/sagemaker-user/GR00T_

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:transformers_modules.Eagle-Block2A-2B-v2.processing_eagle3_vl:Processor Eagle3_VLProcessor:
- image_processor: Eagle3_VLImageProcessorFast {
  "auto_map": {
    "AutoImageProcessor": "image_processing_eagle3_vl_fast.Eagle3_VLImageProcessorFast",
    "AutoProcessor": "processing_eagle3_vl.Eagle3_VLProcessor"
  },
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": false,
  "device": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": false,
  "do_rescale": true,
  "do_resize": false,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Eagle3_VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "input_data_format": null,
  "processor_class": "Eagle3_VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "height": 448,
    "width": 448
  }
}

- tokenizer: Qwen2TokenizerFast(name_or_path='/home/sagemaker-user/GR00T_

Base model evaluation complete.


## 5. Create Held-Out Test Dataset & Evaluate Generalization

Creates 100 test episodes from BridgeData indices 600-699 (never seen during training).

In [ ]:
# Run the test dataset creation script
# This reuses the same logic from 01_data_preparation but for episodes 600-699
import subprocess
TEST_DATASET = os.path.abspath('./datasets/bridge_lerobot_test')
subprocess.run(f'python scripts/utils/create_groot_test_dataset.py --output-path "{TEST_DATASET}"', shell=True, check=True)
print(f'Test dataset created at: {TEST_DATASET}')
print(f'Exists: {os.path.isdir(TEST_DATASET)}')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.2.0 which is incompatible.
autogluon-multimodal 1.5.0 requires Pillow<12,>=10.0.1, but you have pillow 12.2.0 which is incompatible.
autogluon-multimodal 1.5.0 requires scikit-image<0.26.0,>=0.19.1, but you have scikit-image 0.26.0 which is incompatible.
jupyter-scheduler 2.11.0 requires fsspec!=2025.3.1,<=2025.3.2,>=2023.6.0, but you have fsspec 2026.2.0 which is incompatible.
jupyter-scheduler 2.11.0 requires psutil~=5.9, but you have psutil 7.2.2 which is incompa

=== Downloading BridgeData V2 Scripted Images ===
Source: 8802 episodes
Test set: episodes 600 to 699 (100 episodes)


Processing test episodes:   8%|▊         | 8/100 [00:43<08:14,  5.37s/it]

In [ ]:
import os
os.environ["CHECKPOINT_PATH"] = CHECKPOINT_PATH
os.environ["TEST_DATASET"] = TEST_DATASET
os.environ["TRAJ_IDS"] = TRAJ_IDS
os.environ["ACTION_HORIZON"] = str(ACTION_HORIZON)


In [ ]:
%%bash -s "$CHECKPOINT_PATH" "$TEST_DATASET" "$TRAJ_IDS" "$ACTION_HORIZON"
cd Isaac-GR00T

echo "=== Evaluating fine-tuned model on TEST data ==="
python gr00t/eval/open_loop_eval.py \
    --dataset-path "$2" \
    --embodiment-tag NEW_EMBODIMENT \
    --model-path "$1" \
    --traj-ids $3 \
    --action-horizon $4

echo "=== Done ==="

In [ ]:
# Base model on test set
sys.argv = [
    "open_loop_eval.py",
    "--dataset-path", TEST_DATASET,
    "--embodiment-tag", "NEW_EMBODIMENT",
    "--model-path", "nvidia/GR00T-N1.6-3B",
    "--traj-ids",
] + traj_ids + [
    "--action-horizon", str(ACTION_HORIZON),
]

print("Running base model evaluation on test set...")
runpy.run_path("Isaac-GR00T/gr00t/eval/open_loop_eval.py", run_name="__main__")
print("Done.")

## 6. Results Summary

Fill in the numbers from the evaluation outputs above.

### Expected Results (from our run: 600 episodes, 2,000 steps, 8× L40S)

| Metric | Base Model | Fine-Tuned | Improvement |
|--------|-----------|------------|-------------|
| MSE (train) | 1.031e-06 | 5.983e-09 | 172× smaller (99.4% ↓) |
| MAE (train) | 3.782e-04 | 3.103e-05 | 12.2× smaller (91.8% ↓) |
| MSE (test)  | 1.066e-06 | 5.318e-09 | 200× smaller (99.5% ↓) |
| MAE (test)  | 3.766e-04 | 3.008e-05 | 12.5× smaller (92.0% ↓) |

Test set performance matches training set → model generalizes, doesn't just memorize.